# M4

In [ ]:
import os
import sys
from pathlib import Path

for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"

ROOT = Path.cwd().resolve()
while not (ROOT / "utils.py").is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Open this notebook inside the code folder.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from simulation.run_simulation import SimulationConfig, run_simulation
from utils import ratio_table
from IPython.display import HTML


In [ ]:
n = 500  # Number of labeled observations (X and y).
N = 5000  # Number of unlabeled observations (X only).
reps = 100  # Number of repeated experiments for each setting.
dimensions = range(2, 16)  # Numbers of input columns to test, from 2 to 15.
seed = 1  # Random seed for the first repetition.
noise_sigma = 1.0  # Standard deviation of random noise added to the response.
jobs = 12  # Maximum number of repetitions run at the same time.
estimators = [  # Methods to fit to each repeated sample.
    "ata_sbf", "ata_csbf",
    "lp_nw", "lp_ll",  # n^(-k/(21*d)), k = 1, ..., 20, capped at 0.8. Repeated values are used once.
    "krr_m", "krr_r",
    "ease_nw", "ease_ll", "ease_m", "ease_r", "pi",
]

# EASE splits the labeled sample into K groups.
# Fit on K - 1 groups and predict the remaining group.
K = 5
# EASE uses SS-SIR to combine the input columns into r new variables.
# Fit the regression using these r variables.
r = 2
# EASE's SS-SIR splits the response range into H equal-width intervals (slices).
H = n // 5
# Split the labeled sample into cv_folds groups to compare tuning candidates.
# For EASE, fit the full estimator on all but one group for each candidate.
# Use the final EASE coefficients to measure prediction error on the remaining group.
cv_folds = 10

experiment_m4 = SimulationConfig(
    model="M4",
    estimators=estimators,
    dimensions=dimensions,
    n=n,
    N=N,
    reps=reps,
    seed=seed,
    noise_sigma=noise_sigma,
    K=K,
    r=r,
    H=H,
    cv_folds=cv_folds,
    jobs=jobs,
    run_name=f"M4_n{n}_N{N}",
)

summary_m4 = run_simulation(experiment_m4)[0]

In [ ]:
table_m4 = ratio_table(summary_m4, experiment_m4.model)
shown_m4 = table_m4.map(lambda value: f"{value:.3f}")
shown_m4 = shown_m4.mask(
    table_m4.eq(table_m4.max()), "<b>" + shown_m4 + "</b>"
)
HTML(shown_m4.to_html(escape=False))